### 자연어 처리 예제

이 노트북은 **`감정 분석`, `텍스트 요약`, `문서 분류`, `STT(Speech To Text)`, `TTS(Text To Speech)` 기초 예제 실습**을 수행하는 노트북입니다.

✅ 내용  
1. KoBERT를 활용한 감정 분석 예제
2. KoBART를 활용한 텍스트 요약 예제
3. KLUE-RoBERTa를 활용한 문서 분류 예제
4. Whisper를 활용한 STT 예제
5. MMS-TTS, Coqui-TSS+XTTS-v2를 활용한 TTS 예제

In [ ]:
%conda env create -f nlp.yaml -n nlp

### KoBERT를 활용한 감정 분석 예제

사전 학습된 감정분석 모델(`alsgyu/sentiment-analysis-fine-tuned-model`)을 사용하여 입력 텍스트의 감정을 분석합니다.

텍스트를 긍정, 부정, 중립으로 분류하고 각 예측의 신뢰도를 함께 제공합니다.

* 사전 학습된 감정분석 모델과 토크나이저 로드 및 GPU 설정
* 분석할 텍스트 샘플 준비
* 토크나이저를 사용하여 텍스트 인코딩 (패딩, 트런케이션 포함)
* 모델 추론을 통해 감정 확률 계산 및 최고 확률 감정 선택
* 감정 레이블(negative/neutral/positive)과 신뢰도 결과 출력

In [1]:
# ─────────────────────────────────────────────────────────────
# 라이브러리 임포트
# ─────────────────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ─────────────────────────────────────────────────────────────
# 1) 사전 학습된 모델과 토크나이저 로드
# ─────────────────────────────────────────────────────────────
model_name = "alsgyu/sentiment-analysis-fine-tuned-model"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSequenceClassification.from_pretrained(model_name)

# ─────────────────────────────────────────────────────────────
# 2) 디바이스 설정 (GPU 사용 가능 시 GPU로 이동)
# ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ─────────────────────────────────────────────────────────────
# 3) 분석할 텍스트 샘플 준비
# ─────────────────────────────────────────────────────────────
texts = [
    "이 제품은 별로예요.",
    "서비스가 훌륭합니다!",
    "보통이에요."
]

# ─────────────────────────────────────────────────────────────
# 4) 토크나이저로 입력 인코딩 (패딩·트런케이션 포함)
# ─────────────────────────────────────────────────────────────
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
).to(device)

# ─────────────────────────────────────────────────────────────
# 5) 모델 추론 및 확률 계산
# ─────────────────────────────────────────────────────────────
with torch.no_grad():
    outputs = model(**inputs)
    logits  = outputs.logits                     # shape: (batch_size, num_labels)
    probs   = torch.softmax(logits, dim=-1)      # 확률 분포로 변환
    confs, preds = torch.max(probs, dim=-1)      # 최고 확률과 인덱스

# ─────────────────────────────────────────────────────────────
# 6) 레이블 맵핑 및 결과 출력
# ─────────────────────────────────────────────────────────────
labels = ["negative", "neutral", "positive"]

for text, pred, conf in zip(texts, preds.tolist(), confs.tolist()):
    print(f"Input: {text}")
    print(f"  → Sentiment : {labels[pred]}")
    print(f"  → Confidence: {conf:.2f}\n")

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\SSAFY\miniforge3\envs\nlp\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--alsgyu--sentiment-analysis-fine-tuned-model. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Input: 이 제품은 별로예요.
  → Sentiment : negative
  → Confidence: 0.79

Input: 서비스가 훌륭합니다!
  → Sentiment : positive
  → Confidence: 0.99

Input: 보통이에요.
  → Sentiment : positive
  → Confidence: 0.79



### KoBART를 활용한 텍스트 요약 예제

사전 학습된 한국어 요약 모델(`gogamza/kobart-summarization`)을 사용하여 긴 텍스트를 간결하게 요약합니다.

SSAFY 프로그램 소개 문서를 예시로 활용하여 핵심 내용만 추출하는 요약 기능을 구현합니다.

* GPU 설정 및 한국어 요약 파이프라인 초기화
* 요약할 SSAFY 프로그램 소개 텍스트 준비
* 텍스트 요약 실행 (최대 길이 150토큰으로 제한)
* 원문과 요약문 결과 비교 출력

In [2]:
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import re

device = 0 if torch.cuda.is_available() else -1
# ─────────────────────────────────────────────────────────────
# 1) 요약 파이프라인 초기화 (GPU 사용 시 device=0 지정)
# ─────────────────────────────────────────────────────────────
model_name = "gogamza/kobart-summarization"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

summarizer = pipeline(
    "summarization",
    model=model,
    tokenizer=tokenizer,
    device=device  # GPU가 없으면 device=-1로 변경
)

# ─────────────────────────────────────────────────────────────
# 2) 요약할 SSAFY 모집·교육 프로그램 소개 텍스트
# ─────────────────────────────────────────────────────────────
article = """
삼성전자의 소프트웨어 인재 양성 프로그램 SSAFY는 4월 28일부터 5월 12일까지 14기 교육생을 모집합니다.

SSAFY 14기는 2025년 7월부터 전국 5개 캠퍼스에서 1년 간의 교육 과정을 시작할 예정입니다.

"삼성청년SW·AI아카데미"로의 변신을 통한 AI교육 강화

SSAFY는 명칭을 기존 삼성청년SW아카데미에서 삼성청년SW·AI아카데미로 변경하고, AI 과목 신설 등
교육 과정을 전면 개편합니다.
1,2학기 합계 교육시간을 1,620 시간에서 1,725 시간으로 확대하고, AI모델에 대한 이해 및 sLLM 구축 등
수준별 AI강의 및 프로젝트를 신설합니다.
이를 통해 교육생들은 AI를 활용할 수 있는 실전형 SW 개발자로 성장합니다.

"대한민국 취업사관학교" SSAFY

SSAFY는 2018년 12월 1기 교육을 시작한 이래 13기까지 11,750 명의 청년들에게 교육 기회를 제공하였고,
총 7,000명 이상이 1,700여개 기업에 취업했으며, 수료생(1~10기) 기준 취업률은 84%를 기록했습니다.
특히, 교육기회 균등을 위해 2025년부터는 마이스터고 졸업생에게도 문호를 확대하여 교육을 운영 중입니다.

교육생 전원에게 매월 100만 원의 교육 지원비와 전문적인 SW·AI 교육, 전담 취업 컨설턴트의 취업지원 서비스가
무상으로 제공되는 SSAFY!
국내 대표 SW 인재 육성 프로그램을 넘어 AI 사관학교로 발전할 SSAFY와 함께 개발자의 꿈을 체계적으로
키워나가고 싶다면 지금 바로 도전하세요!
"""

# ─────────────────────────────────────────────────────────────
# 3) 텍스트 요약 실행
# ─────────────────────────────────────────────────────────────
summary = summarizer(
    article,
    max_length=150,                       # 요약문 최대 길이
    clean_up_tokenization_spaces=True
)[0]["summary_text"]

# ─────────────────────────────────────────────────────────────
# 4) 결과 출력
# ─────────────────────────────────────────────────────────────
print("=== 원문 ===")
print(article.strip(), "\n")
print("=== 요약문 ===")
print(summary)

config.json: 0.00B [00:00, ?B/s]

c:\Users\SSAFY\miniforge3\envs\nlp\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--gogamza--kobart-summarization. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}.

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels will be overwritten to 2.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels will be overwritten to 2.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Device set to use cuda:0


=== 원문 ===
삼성전자의 소프트웨어 인재 양성 프로그램 SSAFY는 4월 28일부터 5월 12일까지 14기 교육생을 모집합니다.

SSAFY 14기는 2025년 7월부터 전국 5개 캠퍼스에서 1년 간의 교육 과정을 시작할 예정입니다.

"삼성청년SW·AI아카데미"로의 변신을 통한 AI교육 강화

SSAFY는 명칭을 기존 삼성청년SW아카데미에서 삼성청년SW·AI아카데미로 변경하고, AI 과목 신설 등
교육 과정을 전면 개편합니다.
1,2학기 합계 교육시간을 1,620 시간에서 1,725 시간으로 확대하고, AI모델에 대한 이해 및 sLLM 구축 등
수준별 AI강의 및 프로젝트를 신설합니다.
이를 통해 교육생들은 AI를 활용할 수 있는 실전형 SW 개발자로 성장합니다.

"대한민국 취업사관학교" SSAFY

SSAFY는 2018년 12월 1기 교육을 시작한 이래 13기까지 11,750 명의 청년들에게 교육 기회를 제공하였고,
총 7,000명 이상이 1,700여개 기업에 취업했으며, 수료생(1~10기) 기준 취업률은 84%를 기록했습니다.
특히, 교육기회 균등을 위해 2025년부터는 마이스터고 졸업생에게도 문호를 확대하여 교육을 운영 중입니다.

교육생 전원에게 매월 100만 원의 교육 지원비와 전문적인 SW·AI 교육, 전담 취업 컨설턴트의 취업지원 서비스가
무상으로 제공되는 SSAFY!
국내 대표 SW 인재 육성 프로그램을 넘어 AI 사관학교로 발전할 SSAFY와 함께 개발자의 꿈을 체계적으로
키워나가고 싶다면 지금 바로 도전하세요! 

=== 요약문 ===
SSA삼성전자의 소프트웨어 인재 양성 프로그램 SSAFU는 4월 28일부터 5월 12일까지 14기 교육생을 모집하는 SSAFU 14기 교육생을 모집하는 SSAFD 14기는 2025년 7월부터 전국 5개 캠퍼스에서 1년 간의 교육 과정을 시작할 예정이다.


### KLUE-RoBERTa를 활용한 문서 분류 예제

RoBERTa의 한국어 사전 학습 모델을 파인 튜닝한 (`pongjin/roberta_with_kornli`)을 사용하여 텍스트를 분류하는 코드입니다.

미리 정의된 후보 라벨들(정치, 경제, 사회 등) 중에서 입력 텍스트가 어느 카테고리에 속하는지 자동으로 판별합니다.

* GPU/CPU 디바이스 설정 및 커스텀 ArgumentHandler 클래스 정의
* KorNLI 기반 RoBERTa 모델을 사용한 zero-shot 분류 파이프라인 초기화
* 후보 라벨과 가설 템플릿("이는 {}에 관한 내용이다.") 설정
* 입력 텍스트에 대해 zero-shot 분류 실행
* 예측된 라벨 순위와 각 라벨별 신뢰도 점수 출력

In [3]:
from transformers import pipeline, AutoTokenizer  # Hugging Face 파이프라인 유틸리티
import torch                        # PyTorch 텐서 연산

# ─────────────────────────────────────────────────────────────
# 1) 디바이스(Device) 설정
#    - GPU가 사용 가능하면 'cuda:0'(device=0), 그렇지 않으면 CPU(device=-1)
# ─────────────────────────────────────────────────────────────
device = 0 if torch.cuda.is_available() else -1

# ─────────────────────────────────────────────────────────────
# 2) 토크나이저 먼저 로드 (ArgumentHandler에서 사용하기 위함)
# ─────────────────────────────────────────────────────────────
model_name = "pongjin/roberta_with_kornli"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ─────────────────────────────────────────────────────────────
# 3) ArgumentHandler 추상 클래스 정의
#    - 파이프라인에 전달되는 인자를 가공할 기본 인터페이스
# ─────────────────────────────────────────────────────────────
from abc import ABC, abstractmethod

class ArgumentHandler(ABC):
    """
    Pipeline 인자를 전처리하는 추상 인터페이스.
    __call__ 메서드를 구현해 주세요.
    """
    @abstractmethod
    def __call__(self, *args, **kwargs):
        raise NotImplementedError()

# ─────────────────────────────────────────────────────────────
# 4) CustomZeroShotClassificationArgumentHandler 정의
#    - zero-shot 분류를 위해 (premise, hypothesis) 페어를 만들어 줌
#    - 토크나이저를 초기화 시점에 전달받아 사용
# ─────────────────────────────────────────────────────────────
class CustomZeroShotClassificationArgumentHandler(ArgumentHandler):
    """
    zero-shot-classification 파이프라인에 레이블을 NLI 형식으로 변환해 제공
    """
    
    def __init__(self, tokenizer):
        """
        토크나이저를 초기화 시점에 받아서 저장
        """
        self.tokenizer = tokenizer

    def _parse_labels(self, labels):
        # 문자열로 들어오면 쉼표로 분할해 리스트로 변환
        if isinstance(labels, str):
            labels = [lbl.strip() for lbl in labels.split(",")]
        return labels

    def __call__(self, sequences, labels, hypothesis_template):
        # 1) 최소 1개 이상의 시퀀스와 레이블이 있어야 함
        if not labels or not sequences:
            raise ValueError("레이블과 시퀀스를 최소 하나씩 입력해야 합니다.")

        # 2) hypothesis_template에 레이블이 적용되는지 확인
        if hypothesis_template.format(labels[0]) == hypothesis_template:
            raise ValueError(
                f"템플릿 '{hypothesis_template}'에 '{{}}'가 포함되어 있는지 확인하세요."
            )

        # 3) sequences가 단일 문자열이면 리스트로 변환
        if isinstance(sequences, str):
            sequences = [sequences]
        # 4) 레이블도 리스트 형태로 정리
        labels = self._parse_labels(labels)

        # 5) RoBERTa는 token_type_ids를 사용하지 않으므로, SEP 토큰으로 두 문장을 이어서 입력
        sequence_pairs = []
        for label in labels:
            pair = f"{sequences[0]} {self.tokenizer.sep_token} {hypothesis_template.format(label)}"
            sequence_pairs.append(pair)

        # 6) 파이프라인에 넘길 두 값:
        #    - sequence_pairs: 모델에 입력할 NLI 문장 쌍 리스트
        #    - sequences: 원본 시퀀스 리스트
        return sequence_pairs, sequences

# ─────────────────────────────────────────────────────────────
# 5) KorNLI 기반 RoBERTa zero-shot 모델 로드
#    - pongjin/roberta_with_kornli: 한국어 NLI 데이터로 fine-tune된 모델
#    - RoBERTa는 token_type_ids를 쓰지 않으므로 args_parser 지정
# ─────────────────────────────────────────────────────────────
classifier = pipeline(
    "zero-shot-classification",
    model=model_name,
    tokenizer=tokenizer,
    device=device,
    args_parser=CustomZeroShotClassificationArgumentHandler(tokenizer)
)

# ─────────────────────────────────────────────────────────────
# 5) 분류할 후보 라벨과 가설 템플릿, 테스트 문장 준비
# ─────────────────────────────────────────────────────────────
candidate_labels    = ["정치", "경제", "사회", "생활문화", "세계", "IT과학", "스포츠"]
hypothesis_template = "이는 {}에 관한 내용이다."
sequence           = "배당락 D-1 코스피, 2330선 상승세...외인·기관 사자"

# ─────────────────────────────────────────────────────────────
# 6) zero-shot 분류 실행
#    - classifier(시퀀스, 레이블, 템플릿) 호출
#    - 반환 결과: dict with keys ['sequence', 'labels', 'scores']
# ─────────────────────────────────────────────────────────────
result = classifier(
    sequence,
    candidate_labels,
    hypothesis_template=hypothesis_template
)

# ─────────────────────────────────────────────────────────────
# 7) 결과 출력
# ─────────────────────────────────────────────────────────────
print("입력 텍스트:", result["sequence"])
print("→ 예측 순위:", result["labels"])
print("→ 신뢰도 점수:", [round(score, 3) for score in result["scores"]])

tokenizer_config.json:   0%|          | 0.00/415 [00:00<?, ?B/s]

c:\Users\SSAFY\miniforge3\envs\nlp\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--pongjin--roberta_with_kornli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Device set to use cuda:0


입력 텍스트: 배당락 D-1 코스피, 2330선 상승세...외인·기관 사자
→ 예측 순위: ['경제', '정치', 'IT과학', '세계', '사회', '생활문화', '스포츠']
→ 신뢰도 점수: [0.388, 0.119, 0.119, 0.117, 0.109, 0.09, 0.058]


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


### Whisper를 활용한 STT 예제

시스템 마이크를 통해 실시간으로 음성을 녹음하고 OpenAI Whisper 모델(`openai/whisper-small`)을 사용하여 한국어 음성을 텍스트로 변환합니다.

sounddevice 라이브러리로 마이크 입력을 처리하고, Whisper 모델의 한국어 디코딩 설정을 통해 한국어 음성 인식을 수행합니다.

* 필요한 라이브러리 임포트 (sounddevice, soundfile, transformers 등)
* 시스템 마이크로 5초간 16kHz 모노 오디오 녹음
* 녹음된 NumPy 배열을 WAV 파일로 저장
* Whisper small 모델로 ASR 파이프라인 초기화 및 한국어 강제 디코딩 설정
* 저장된 오디오 파일에서 텍스트 추출 및 결과 출력

In [4]:
# ─────────────────────────────────────────────────────────────
# 1) 라이브러리 임포트
# ─────────────────────────────────────────────────────────────
import sounddevice as sd         # 마이크 입력 녹음용
import soundfile as sf           # WAV 파일 저장/불러오기
from transformers import pipeline
import torch

# ─────────────────────────────────────────────────────────────
# 2) 시스템 마이크로 5초간 오디오 녹음
# ─────────────────────────────────────────────────────────────
duration = 5             # 녹음 시간(초)
sampling_rate = 16000    # Whisper 권장 샘플링 레이트(16kHz)
channels = 1             # mono 녹음

print("마이크 녹음 시작...")
# 전체 프레임 수 = duration * sampling_rate
audio_data = sd.rec(int(duration * sampling_rate),
                    samplerate=sampling_rate,
                    channels=channels)
sd.wait()  # 녹음 완료 대기
print("녹음 완료!")

# ─────────────────────────────────────────────────────────────
# 3) 녹음된 NumPy 배열을 WAV 파일로 저장
# ─────────────────────────────────────────────────────────────
wav_path = "mic_input.wav"
# float32 배열을 그대로 WAV로 저장
sf.write(wav_path, audio_data, sampling_rate)
print(f"WAV 파일로 저장: {wav_path}")

# ─────────────────────────────────────────────────────────────
# 4) Whisper small 한국어 ASR 파이프라인 초기화
# ─────────────────────────────────────────────────────────────
device = 0 if torch.cuda.is_available() else -1
asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device
)
# 한국어 강제 디코딩 설정 (언어="ko", 작업="transcribe")
asr.model.config.forced_decoder_ids = (
    asr.tokenizer.get_decoder_prompt_ids(language="ko", task="transcribe")
)

# ─────────────────────────────────────────────────────────────
# 5) 저장된 오디오 파일로부터 텍스트 추출
# ─────────────────────────────────────────────────────────────
result = asr(wav_path)
print("Whisper 결과:", result['text'])

마이크 녹음 시작...
녹음 완료!
WAV 파일로 저장: mic_input.wav


Device set to use cuda:0
c:\Users\SSAFY\.conda\envs\nlp\lib\site-packages\transformers\models\whisper\generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


Whisper 결과:  you


### MMS-TTS를 활용한 TTS 예제

Facebook의 MMS(Massively Multilingual Speech) TTS 모델(`facebook/mms-tts-kor`)을 사용하여 한국어 텍스트를 자연스러운 음성으로 변환하는 코드입니다.

여러 문장을 하나로 결합하여 연속적인 음성을 생성하고, librosa를 활용해 속도와 피치를 조정한 다양한 버전의 오디오를 제작합니다.

* GPU/CPU 디바이스 설정 및 MMS 한국어 TTS 모델과 토크나이저 로드
* uroman을 사용한 한국어 텍스트의 로마자 변환 및 토크나이징
* VITS 모델을 통한 음성 파형 생성 및 원본 음성 저장
* librosa를 활용한 속도 조정(time stretch)과 피치 조정(pitch shift) 적용
* 원본, 빠른 속도, 높은 피치 버전의 오디오를 각각 WAV 파일로 저장 및 Jupyter에서 재생

In [5]:
from transformers import VitsModel, AutoTokenizer
import torch
import soundfile as sf
import uroman as ur
import librosa
from IPython.display import Audio, display

# 1) 디바이스 설정: GPU 사용 가능 시 'cuda', 아니면 'cpu'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2) 로마자 변환기 초기화
romanizer = ur.Uroman()

# 3) MMS 한국어 VITS 모델과 토크나이저 로드 후 디바이스로 이동
model = VitsModel.from_pretrained("facebook/mms-tts-kor").to(device)
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-kor")

# 4) 합성할 문장 3개를 하나의 문자열로 결합
texts = [
    "안녕하세요.",
    "오늘은 날씨가 좋네요.",
    "무엇을 도와드릴까요?"
]
full_text = " ".join(texts)
print("Combined text:", full_text)

# 5) 속도 및 피치 조정 파라미터
speed_rate = 1.2  # 20% 빠르게
pitch_steps = 2   # +2 semitones

# 6) 로마자 변환 → 토크나이즈 → 합성
romanized = romanizer.romanize_string(full_text)
inputs = tokenizer(romanized, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    waveform_tensor = model(**inputs).waveform

# 7) GPU 텐서를 CPU로 옮겨 NumPy 배열로 변환
waveform = waveform_tensor[0].cpu().numpy().T
sr = model.config.sampling_rate

# ─────────────────────────────────────────────────────────────
# A) 원본 음성 저장 및 재생
# ─────────────────────────────────────────────────────────────
orig_path = "combined_orig.wav"
sf.write(orig_path, waveform, sr)
print("Original:")
display(Audio(waveform, rate=sr))

# ─────────────────────────────────────────────────────────────
# B) 속도 조정 저장 및 재생
# ─────────────────────────────────────────────────────────────
fast_wave = librosa.effects.time_stretch(waveform, rate=speed_rate)
fast_path = "combined_fast.wav"
sf.write(fast_path, fast_wave, sr)
print(f"Fast (x{speed_rate}):")
display(Audio(fast_wave, rate=sr))

# ─────────────────────────────────────────────────────────────
# C) 피치 조정 저장 및 재생
# ─────────────────────────────────────────────────────────────
pitch_wave = librosa.effects.pitch_shift(waveform, sr=sr, n_steps=pitch_steps)
pitch_path = "combined_pitch.wav"
sf.write(pitch_path, pitch_wave, sr)
print(f"Pitch (+{pitch_steps}st):")
display(Audio(pitch_wave, rate=sr))


Using device: cuda
Combined text: 안녕하세요. 오늘은 날씨가 좋네요. 무엇을 도와드릴까요?
Original:


Fast (x1.2):


Pitch (+2st):


### Coqui-TSS+XTTS-v2를 활용한 TTS 예제

Coqui TTS의 XTTS-v2 모델(`tts_models/multilingual/multi-dataset/xtts_v2`)을 사용하여 마이크로 녹음한 참조 음성을 기반으로 음성 클로닝을 수행하는 코드입니다.

사용자의 목소리를 10초간 녹음한 후, 해당 음성의 톤과 특성을 학습하여 새로운 한국어 텍스트를 동일한 목소리 스타일로 합성합니다.

* 필수 라이브러리 임포트 (Coqui TTS, PyTorch, sounddevice 등) 및 GPU/CPU 디바이스 설정
* XTTS-v2 다국어 음성 클로닝 모델 로드
* 시스템 마이크로 10초간 22.05kHz 샘플링 레이트로 참조 음성 녹음
* 녹음된 참조 음성을 WAV 파일로 저장
* XTTS 모델을 사용하여 참조 음성 스타일로 한국어 텍스트 음성 합성 및 Jupyter에서 재생

In [6]:
# 1) 필수 모듈 임포트
from TTS.api import TTS           # Coqui TTS의 고수준 API를 사용하기 위한 모듈
import torch                      # PyTorch: 모델 로드 및 연산(텐서 계산) 용도
import sounddevice as sd          # 마이크 녹음 및 오디오 스트림 제어를 위한 모듈
import soundfile as sf            # 오디오 파일 읽기/쓰기(예: WAV 파일 저장, 로드)
from IPython.display import Audio, display  
                                 # Jupyter Notebook 환경에서 오디오 재생을 위해 필요

# 2) 디바이스 설정: CUDA가 사용 가능하면 GPU('cuda')를, 그렇지 않으면 CPU('cpu')를 사용
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # 선택된 디바이스(예: cuda 또는 cpu)를 콘솔에 출력하여 확인

# 3) XTTS-v2 모델 로드: 주어진 참조 음성 스타일대로 음성 클로닝을 수행하는 모델
print("Loading XTTS model...")
try:
    # TTS 클래스에 모델 이름을 전달하여 사전 학습된 XTTS-v2 모델을 로드
    tts = TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2")
    # .to(device)를 호출하여 모델을 앞서 설정한 디바이스(CUDA 또는 CPU)로 이동
    tts.to(device)
    print("XTTS model loaded successfully.")  # 모델 로드가 완료되었음을 출력
except Exception as e:
    # 모델 로드 중 예외가 발생하면 에러 메시지를 출력하고 예외를 다시 발생시킴
    print(f"Error loading TTS model: {e}")
    raise

# 4) 클로닝에 사용할 참조 음성 샘플과 합성할 텍스트 정의
#    - speaker_wav_path: 사용자가 녹음하거나 미리 준비한 3~6초 길이의 한국어 음성 파일 경로
#    - text_to_clone: 위 참조 음성 스타일로 합성할 한국어 문장
#    - output_wav_path: 합성된 음성을 저장할 출력 파일 이름/경로

# 5) 마이크를 사용하여 음성 녹음 (선택적)
#    duration: 녹음 시간(초 단위)
duration = 10               # 녹음 시간(초) 설정
sr_record = 22050           # 녹음 샘플링 레이트(Hz) 설정 (예: 22050Hz)

print(f"Recording {duration} seconds from microphone…")
# sd.rec: 마이크로부터 duration * sr_record 길이만큼의 오디오 데이터를 녹음
#   samplerate: 녹음 샘플링 레이트 (sr_record)
#   channels=1: 모노(단일 채널) 녹음
recording = sd.rec(int(duration * sr_record), samplerate=sr_record, channels=1)
sd.wait()                   # sd.wait(): 녹음이 완료될 때까지 대기

# 녹음된 오디오 데이터를 "mic_speaker.wav" 파일로 저장
speaker_wav_path = "mic_speaker.wav"
# 만약 이미 녹음된 파일을 사용하여 클로닝을 하고 싶다면 아래와 같이 주석 해제
# speaker_wav_path = "8aab6749.wav"              

# sf.write: NumPy 배열 형태의 recording 데이터를 WAV 파일로 저장
#   첫 번째 인자: 파일명 (speaker_wav_path)
#   두 번째 인자: 녹음된 오디오 데이터 (NumPy 배열)
#   세 번째 인자: 샘플링 레이트 (sr_record)
sf.write(speaker_wav_path, recording, sr_record)
print(f"Microphone audio saved to '{speaker_wav_path}'")

# 6) 클로닝 대상 텍스트 및 출력 파일 경로 지정
text_to_clone   = "이것은 제 목소리로 복제된 한국어 음성입니다. 잘 들어보세요."
output_wav_path = "cloned_korean_tts.wav" 

try:
    # 7) TTS 수행: 참조 음성 스타일로 텍스트를 음성으로 변환하여 파일로 저장
    # tts.tts_to_file: 텍스트와 참조 음성 샘플을 함께 전달하여
    #                  해당 화자의 목소리에 가깝게 음성을 합성
    tts.tts_to_file(
        text=text_to_clone,         # 합성할 한국어 텍스트
        speaker_wav=speaker_wav_path,# 참조할 음성 샘플 파일 경로
        language="ko",               # 언어 코드("ko"는 한국어)
        file_path=output_wav_path    # 합성된 음성을 저장할 파일 경로
    )
    print(f"Cloned speech saved to '{output_wav_path}'")

    # 8) 저장된 WAV 파일 읽기
    # sf.read: 오디오 파일을 읽어서 NumPy 배열(audio_data)과 샘플링 레이트(sr)를 반환
    audio_data, sr = sf.read(output_wav_path)  
    #   audio_data: NumPy 배열 형태로 저장된 음성 데이터
    #   sr: 샘플링 레이트 (Hz)

    # 9) Jupyter Notebook 셀에서 바로 재생
    print("Playing cloned speech:")
    # Audio: IPython.display.Audio 객체를 생성하여 오디오를 재생
    #   첫 번째 인자: 재생할 오디오 데이터 (NumPy 배열)
    #   rate: 재생 속도(샘플링 레이트)
    display(Audio(audio_data, rate=sr))  

except Exception as e:
    # 합성 중 예외가 발생하면 에러 메시지를 출력
    print(f"Error during TTS synthesis: {e}")

Using device: cuda
Loading XTTS model...
XTTS model loaded successfully.
Recording 10 seconds from microphone…
Microphone audio saved to 'mic_speaker.wav'
Cloned speech saved to 'cloned_korean_tts.wav'
Playing cloned speech:
